In [2]:
import random
import operator
import math
import numpy as np
from deap import base, creator, tools

In [1]:
# Number of dimensions in each particle's position vector.
NUM_DECISION_VARIABLES = 20

# Number of particles in the swarm and optimization iterations.
POPULATION_SIZE = 50
NUM_OF_GEN = 1000

# Bound each velocity component to keep particle movement stable.
VELOCITY_MIN = -3
VELOCITY_MAX = 3

# Bound each decision variable within the search space.
POS_MIN = -30
POS_MAX = 30

# Acceleration coefficients in the PSO update.
PHI1_CONSTANT = 1.49618
PHI2_CONSTANT = 1.49618

# Controls how much of the previous velocity carries into the next update.
INERTIA_WEIGHT = 0.7298

In [141]:
# Create a particle with a random position and velocity in the search bounds.
def generate_particle(size, pos_min, pos_max, vel_min, vel_max):
    particle = creator.Particle(random.uniform(pos_min, pos_max) for _ in range(size))
    particle.velocity = [random.uniform(vel_min, vel_max) for _ in range(size)]
    particle.vel_min = vel_min
    particle.vel_max = vel_max
    return particle


def update_particle(particle, best, phi_1, phi_2, inertia_weight):
    # Draw independent random acceleration factors for each dimension.
    u1 = (random.uniform(0, phi_1) for _ in range(len(particle)))
    u2 = (random.uniform(0, phi_2) for _ in range(len(particle)))

    # The cognitive term pulls the particle toward its personal best.
    cognitive_component = map(
        operator.mul,
        u1,
        map(operator.sub, particle.best, particle)
    )

    # The social term pulls the particle toward the swarm's global best.
    social_component = map(
        operator.mul,
        u2,
        map(operator.sub, best, particle)
    )

    # Combine inertia, personal experience, and swarm experience to update velocity.
    particle.velocity = list(map(
        operator.add,
        (inertia_weight * velocity for velocity in particle.velocity),
        map(operator.add, cognitive_component, social_component)
    ))

    # Clip each velocity component so a particle cannot move too far in one step.
    for i, velocity in enumerate(particle.velocity):
        if abs(velocity) > particle.vel_max:
            particle.velocity[i] = math.copysign(
                particle.vel_max, velocity
            )

    # Move the particle using its newly calculated velocity.
    particle[:] = list(map(
        operator.add,
        particle,
        particle.velocity
    ))

    # Clip each position component to the permitted search space.
    for i, position in enumerate(particle):
        if position < POS_MIN:
            particle[i] = POS_MIN
        elif position > POS_MAX:
            particle[i] = POS_MAX


# Evaluate a particle using the selected objective function.
def evaluation_rosenbrock(individual):
    fitness = sum(
        100 * (individual[i] ** 2 - individual[i + 1]) ** 2
        + (individual[i] - 1) ** 2
        for i in range(len(individual) - 1)
    )
    return (fitness,)


def evaluation_griewank(individual):
    sum_part = sum(
        x ** 2 / 4000
        for x in individual
    )

    product_part = math.prod(
        math.cos(x / math.sqrt(i + 1))
        for i, x in enumerate(individual)
    )

    fitness = sum_part - product_part + 1

    return (fitness,)

In [147]:
# Define a minimization fitness and the particle data structure.
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Particle", list, fitness=creator.FitnessMin, velocity=list, vel_min=None, vel_max=None, best=None)

# Register the particle factory and PSO operations with the DEAP toolbox.
toolbox = base.Toolbox()

toolbox.register("particle",
                 generate_particle,
                 size=NUM_DECISION_VARIABLES,
                 pos_min=POS_MIN,
                 pos_max=POS_MAX,
                 vel_min=VELOCITY_MIN,
                 vel_max=VELOCITY_MAX)

# Build a swarm by repeatedly calling the particle factory.
toolbox.register("population", tools.initRepeat, list, toolbox.particle)

# Bind the PSO coefficients to the particle update function.
toolbox.register("update",
                 update_particle,
                 phi_1=PHI1_CONSTANT,
                 phi_2=PHI2_CONSTANT,
                 inertia_weight=INERTIA_WEIGHT)

# Select the objective function used to assign each particle's fitness.
toolbox.register("evaluate", evaluation_griewank)

/opt/anaconda3/lib/python3.13/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMin' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/opt/anaconda3/lib/python3.13/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Particle' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


In [148]:
# Fix the seed so that the experiment is reproducible.
random.seed(512)

population = toolbox.population(POPULATION_SIZE)
best = None

for gen in range(NUM_OF_GEN):
    # Evaluate the current swarm and refresh personal and global bests.
    for part in population:
        part.fitness.values = toolbox.evaluate(part)

        # Store a snapshot when this particle improves its personal best.
        if not part.best or part.best.fitness < part.fitness:
            part.best = creator.Particle(part)
            part.best.fitness.values = part.fitness.values

        # Store a snapshot when this particle improves the global best.
        if not best or best.fitness < part.fitness:
            best = creator.Particle(part)
            best.fitness.values = part.fitness.values

    # Use the best positions found so far to update every particle.
    for part in population:
        toolbox.update(part, best)

In [149]:
print("Best solution:", best)
print("Minimum fitness:", best.fitness.values[0])

Best solution: [3.140022611640793, -1.2858692588416786e-08, -1.2349933219047236e-08, -6.270643927918053, 1.8746127887143807e-08, 8.31102365544546e-09, -1.4585796131251119e-08, -5.897269760460196e-09, -1.156105614701206e-08, -2.8581089951522516e-08, 2.3452664300213223e-08, 3.607289541809148e-08, -1.9819507045512248e-08, 1.5178559423047957e-08, -4.114873668173777e-09, 1.896719551929324e-08, -3.829807663267509e-10, 1.1113031480515116e-08, 1.7789172751850832e-08, -4.533387089107881e-08]
Minimum fitness: 0.01231607251926159


In [150]:
print("Fitness:", best.fitness.values[0])
print("Min position:", min(best))
print("Max position:", max(best))

Fitness: 0.01231607251926159
Min position: -6.270643927918053
Max position: 3.140022611640793


In [151]:
toolbox.evaluate(best)

(0.01231607251926159,)